# Graph Convolutional Networks on the Cora dataset
In this notebook, we demonstrate the use of graph techniques for **node classification**.

The Cora dataset is a graph dataset where the nodes are the scientific papers and the edges are citation links between papers. In the (planetoid) version of Cora dataset used here, the links are *undirected*. The nodes have multiple types

Here, we are trying to predict the types of nodes in the graph. There are 7 different possible types of scientific papers.

## Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.datasets import Planetoid
import torch_geometric.nn as pyg_nn

import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

## Load data

In [2]:
# load Cora dataset
dataset = Planetoid(root="/tmp/Cora", name="Cora")
data = dataset[0]  # single graph object

print("Dataset:", dataset)
print("Graph:", data)
print("Number of nodes:", data.num_nodes)
print("Number of edges:", data.num_edges)
print("Number of features:", dataset.num_node_features)
print("Number of classes:", dataset.num_classes)

Dataset: Cora()
Graph: Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
Number of nodes: 2708
Number of edges: 10556
Number of features: 1433
Number of classes: 7


## 1. Graph Convolutional Network (GCN)

In [3]:
# GCN model
class GCN(nn.Module):
    def __init__(self, in_feats, hidden_feats, num_classes):
        super(GCN, self).__init__()
        self.conv1 = pyg_nn.GCNConv(in_feats, hidden_feats)
        self.conv2 = pyg_nn.GCNConv(hidden_feats, num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        # x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GCN(dataset.num_node_features, 16, dataset.num_classes).to(device)
data = data.to(device)

In [4]:
# loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

In [5]:
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])  # use only train nodes
    loss.backward()
    optimizer.step()
    return loss.item()

def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    
    accs = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        correct = (pred[mask] == data.y[mask]).sum()
        acc = int(correct) / int(mask.sum())
        accs.append(acc)
    return accs

In [6]:
# TRAIN!
for epoch in range(200):
    loss = train()
    train_acc, val_acc, test_acc = test()
    if epoch % 20 == 0:
        print(f"Epoch {epoch:03d}, Loss: {loss:.4f}, "
              f"Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}")

# Final test accuracy
_, _, test_acc = test()
print("Final Test Accuracy:", test_acc)

Epoch 000, Loss: 1.9622, Train Acc: 0.6071, Val Acc: 0.4180, Test Acc: 0.4210
Epoch 020, Loss: 0.1602, Train Acc: 0.9857, Val Acc: 0.7760, Test Acc: 0.7890
Epoch 040, Loss: 0.0184, Train Acc: 1.0000, Val Acc: 0.7800, Test Acc: 0.7960
Epoch 060, Loss: 0.0160, Train Acc: 1.0000, Val Acc: 0.7720, Test Acc: 0.7950
Epoch 080, Loss: 0.0184, Train Acc: 1.0000, Val Acc: 0.7780, Test Acc: 0.8010
Epoch 100, Loss: 0.0171, Train Acc: 1.0000, Val Acc: 0.7740, Test Acc: 0.7970
Epoch 120, Loss: 0.0151, Train Acc: 1.0000, Val Acc: 0.7720, Test Acc: 0.8020
Epoch 140, Loss: 0.0137, Train Acc: 1.0000, Val Acc: 0.7720, Test Acc: 0.8030
Epoch 160, Loss: 0.0126, Train Acc: 1.0000, Val Acc: 0.7700, Test Acc: 0.8030
Epoch 180, Loss: 0.0117, Train Acc: 1.0000, Val Acc: 0.7640, Test Acc: 0.8040
Final Test Accuracy: 0.806


In [7]:
def evaluate(title=''):
    # Class names for Cora dataset
    class_names = [
        "Case_Based",
        "Genetic_Algorithms",
        "Neural_Networks",
        "Probabilistic_Methods",
        "Reinforcement_Learning",
        "Rule_Learning",
        "Theory"
    ]
    
    # Get predictions for test nodes
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1).cpu().numpy()
    
    y_true = data.y[data.test_mask].cpu().numpy()
    y_pred = pred[data.test_mask.cpu().numpy()]

    # Title
    print('***', title, '***')
    print('')
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    print("Confusion Matrix:\n", cm)
    
    # Classification report with class names
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names))

In [8]:
evaluate('GCN')

*** GCN ***

Confusion Matrix:
 [[ 93   4   3   7   7   6  10]
 [  4  80   3   3   0   0   1]
 [  3   6 130   4   0   1   0]
 [ 21   5   7 244  31   5   6]
 [  9   1   1   7 126   4   1]
 [  9   4   4   0   0  77   9]
 [  4   1   0   1   0   2  56]]

Classification Report:
                        precision    recall  f1-score   support

            Case_Based       0.65      0.72      0.68       130
    Genetic_Algorithms       0.79      0.88      0.83        91
       Neural_Networks       0.88      0.90      0.89       144
 Probabilistic_Methods       0.92      0.76      0.83       319
Reinforcement_Learning       0.77      0.85      0.81       149
         Rule_Learning       0.81      0.75      0.78       103
                Theory       0.67      0.88      0.76        64

              accuracy                           0.81      1000
             macro avg       0.78      0.82      0.80      1000
          weighted avg       0.82      0.81      0.81      1000



## 2. GraphSAGE

In [9]:
class GraphSAGE(nn.Module):
    def __init__(self, in_feats, hidden_feats, num_classes):
        super(GraphSAGE, self).__init__()
        self.sage1 = pyg_nn.SAGEConv(in_feats, hidden_feats, aggr="mean")
        self.sage2 = pyg_nn.SAGEConv(hidden_feats, num_classes, aggr="mean")

    def forward(self, x, edge_index):
        x = self.sage1(x, edge_index)
        x = F.relu(x)
        # x = F.dropout(x, p=0.5, training=self.training)
        x = self.sage2(x, edge_index)
        return x

model = GraphSAGE(dataset.num_node_features, 16, dataset.num_classes).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

In [10]:
# TRAIN!
for epoch in range(200):
    loss = train()
    train_acc, val_acc, test_acc = test()
    if epoch % 20 == 0:
        print(f"Epoch {epoch:03d}, Loss: {loss:.4f}, "
              f"Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}")

# Final test accuracy
_, _, test_acc = test()
print("Final Test Accuracy (GraphSAGE):", test_acc)

Epoch 000, Loss: 1.9523, Train Acc: 0.7286, Val Acc: 0.3080, Test Acc: 0.3260
Epoch 020, Loss: 0.0026, Train Acc: 1.0000, Val Acc: 0.7560, Test Acc: 0.7480
Epoch 040, Loss: 0.0005, Train Acc: 1.0000, Val Acc: 0.7620, Test Acc: 0.7720
Epoch 060, Loss: 0.0018, Train Acc: 1.0000, Val Acc: 0.7580, Test Acc: 0.7800
Epoch 080, Loss: 0.0041, Train Acc: 1.0000, Val Acc: 0.7700, Test Acc: 0.7930
Epoch 100, Loss: 0.0045, Train Acc: 1.0000, Val Acc: 0.7720, Test Acc: 0.7990
Epoch 120, Loss: 0.0042, Train Acc: 1.0000, Val Acc: 0.7780, Test Acc: 0.8040
Epoch 140, Loss: 0.0040, Train Acc: 1.0000, Val Acc: 0.7780, Test Acc: 0.8040
Epoch 160, Loss: 0.0037, Train Acc: 1.0000, Val Acc: 0.7740, Test Acc: 0.8020
Epoch 180, Loss: 0.0035, Train Acc: 1.0000, Val Acc: 0.7760, Test Acc: 0.8050
Final Test Accuracy (GraphSAGE): 0.804


In [11]:
evaluate('GraphSAGE')

*** GraphSAGE ***

Confusion Matrix:
 [[ 95   4   2   7   8   3  11]
 [  3  81   3   2   0   1   1]
 [  2   5 130   5   0   1   1]
 [ 20   9   6 232  35  11   6]
 [  8   1   0   5 128   5   2]
 [  6   2   3   1   0  81  10]
 [  3   1   0   1   0   2  57]]

Classification Report:
                        precision    recall  f1-score   support

            Case_Based       0.69      0.73      0.71       130
    Genetic_Algorithms       0.79      0.89      0.84        91
       Neural_Networks       0.90      0.90      0.90       144
 Probabilistic_Methods       0.92      0.73      0.81       319
Reinforcement_Learning       0.75      0.86      0.80       149
         Rule_Learning       0.78      0.79      0.78       103
                Theory       0.65      0.89      0.75        64

              accuracy                           0.80      1000
             macro avg       0.78      0.83      0.80      1000
          weighted avg       0.82      0.80      0.81      1000



## 3. Graph Attention Networks (GAT)

In [12]:
class GAT(nn.Module):
    def __init__(self, in_feats, hidden_feats, num_classes, heads=8):
        super(GAT, self).__init__()
        # First GAT layer: multi-head attention
        self.gat1 = pyg_nn.GATConv(in_feats, hidden_feats, heads=heads, dropout=0.6)
        # Second GAT layer: aggregate heads into output
        self.gat2 = pyg_nn.GATConv(hidden_feats * heads, num_classes, heads=1, concat=False, dropout=0.6)

    def forward(self, x, edge_index):
        x = self.gat1(x, edge_index)
        x = F.elu(x)  # nonlinearity after attention
        # x = F.dropout(x, p=0.6, training=self.training)
        x = self.gat2(x, edge_index)
        return x

model = GAT(dataset.num_node_features, 8, dataset.num_classes).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

In [13]:
# TRAIN!
for epoch in range(200):
    loss = train()
    train_acc, val_acc, test_acc = test()
    if epoch % 20 == 0:
        print(f"Epoch {epoch:03d}, Loss: {loss:.4f}, "
              f"Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}")

# Final test accuracy
_, _, test_acc = test()
print("Final Test Accuracy (GAT):", test_acc)

Epoch 000, Loss: 1.9555, Train Acc: 0.5500, Val Acc: 0.3780, Test Acc: 0.3930
Epoch 020, Loss: 0.6071, Train Acc: 0.9929, Val Acc: 0.7840, Test Acc: 0.8040
Epoch 040, Loss: 0.3496, Train Acc: 1.0000, Val Acc: 0.7860, Test Acc: 0.8060
Epoch 060, Loss: 0.2150, Train Acc: 1.0000, Val Acc: 0.7780, Test Acc: 0.8060
Epoch 080, Loss: 0.3747, Train Acc: 1.0000, Val Acc: 0.7900, Test Acc: 0.8080
Epoch 100, Loss: 0.3361, Train Acc: 1.0000, Val Acc: 0.7760, Test Acc: 0.8040
Epoch 120, Loss: 0.2253, Train Acc: 1.0000, Val Acc: 0.7780, Test Acc: 0.7960
Epoch 140, Loss: 0.3196, Train Acc: 1.0000, Val Acc: 0.7720, Test Acc: 0.8060
Epoch 160, Loss: 0.4357, Train Acc: 1.0000, Val Acc: 0.7820, Test Acc: 0.7980
Epoch 180, Loss: 0.2184, Train Acc: 1.0000, Val Acc: 0.7800, Test Acc: 0.8020
Final Test Accuracy (GAT): 0.811


In [14]:
evaluate('GAT')

*** GAT ***

Confusion Matrix:
 [[ 98   5   4   8   3   4   8]
 [  4  77   3   5   0   1   1]
 [  1   3 133   5   0   2   0]
 [ 23   9   7 247  21   7   5]
 [  7   3   1  10 121   6   1]
 [  7   3   4   0   0  81   8]
 [  6   1   0   1   0   2  54]]

Classification Report:
                        precision    recall  f1-score   support

            Case_Based       0.67      0.75      0.71       130
    Genetic_Algorithms       0.76      0.85      0.80        91
       Neural_Networks       0.88      0.92      0.90       144
 Probabilistic_Methods       0.89      0.77      0.83       319
Reinforcement_Learning       0.83      0.81      0.82       149
         Rule_Learning       0.79      0.79      0.79       103
                Theory       0.70      0.84      0.77        64

              accuracy                           0.81      1000
             macro avg       0.79      0.82      0.80      1000
          weighted avg       0.82      0.81      0.81      1000

